In [1]:
import os
import json
from collections import Counter

label_dir = r"D:\Dataset\Dental_dataset\Pediatric dental disease detection dataset\Train\label"

label_count_per_image = Counter()
multi_label_files = []

for file in os.listdir(label_dir):

    if not file.endswith(".json"):
        continue

    filepath = os.path.join(label_dir, file)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Unique labels in one image
    image_labels = set()

    for shape in data.get("shapes", []):
        image_labels.add(shape["label"])

    label_count_per_image[len(image_labels)] += 1

    if len(image_labels) > 1:
        multi_label_files.append(
            (file, sorted(image_labels))
        )

print("=" * 50)
print("Unique Disease Classes Per Image")
print("=" * 50)

for n_labels, count in sorted(label_count_per_image.items()):
    print(
        f"Images with {n_labels} unique class(es): {count}"
    )

print("\nTotal multi-label images:", len(multi_label_files))

print("\nSample multi-label images:")

for file, labels in multi_label_files[:10]:
    print(file, "->", labels)

Unique Disease Classes Per Image
Images with 1 unique class(es): 35
Images with 2 unique class(es): 31
Images with 3 unique class(es): 4

Total multi-label images: 35

Sample multi-label images:
cate1_000.json -> ['深窝沟', '龋病']
cate1_002.json -> ['牙髓炎', '龋病']
cate1_004.json -> ['根尖周炎', '牙齿发育异常', '龋病']
cate1_005.json -> ['牙髓炎', '龋病']
cate1_006.json -> ['牙齿发育异常', '龋病']
cate1_007.json -> ['其他', '龋病']
cate1_010.json -> ['深窝沟', '龋病']
cate2_003.json -> ['牙齿发育异常', '龋病']
cate2_004.json -> ['牙齿发育异常', '龋病']
cate2_005.json -> ['深窝沟', '牙髓炎']


In [4]:
# ============================================================
# STEP 1: LIBRARIES + DATASET LOADING + JSON PARSING
# ============================================================

import os
import json
import random
import warnings

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings("ignore")


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. CONFIGURATION
# ============================================================

IMG_SIZE = 224
NUM_CLASSES = 6
BATCH_SIZE = 32


# ============================================================
# 3. CORRECT DATASET PATHS
# ============================================================

BASE_DIR = r"D:\Dataset\Dental_dataset\Pediatric dental disease detection dataset"

TRAIN_IMAGE_DIR = os.path.join(
    BASE_DIR,
    "Train",
    "images"
)

TRAIN_LABEL_DIR = os.path.join(
    BASE_DIR,
    "Train",
    "label"
)

TEST_IMAGE_DIR = os.path.join(
    BASE_DIR,
    "Test",
    "images"
)

TEST_LABEL_DIR = os.path.join(
    BASE_DIR,
    "Test",
    "label"
)


# ============================================================
# 4. VERIFY DATASET PATHS
# ============================================================

print("=" * 70)
print("DATASET PATH VERIFICATION")
print("=" * 70)

print(
    "Train Images:",
    os.path.exists(TRAIN_IMAGE_DIR),
    TRAIN_IMAGE_DIR
)

print(
    "Train Labels:",
    os.path.exists(TRAIN_LABEL_DIR),
    TRAIN_LABEL_DIR
)

print(
    "Test Images:",
    os.path.exists(TEST_IMAGE_DIR),
    TEST_IMAGE_DIR
)

print(
    "Test Labels:",
    os.path.exists(TEST_LABEL_DIR),
    TEST_LABEL_DIR
)


# ============================================================
# 5. CLASS MAPPING
# ============================================================

CLASS_MAPPING = {

    "龋病":
        "Dental Caries",

    "根尖周炎":
        "Periapical Periodontitis",

    "牙髓炎":
        "Pulpitis",

    "深窝沟":
        "Deep Pits & Fissures",

    "牙齿发育异常":
        "Dental Developmental Anomalies",

    "其他":
        "Others"
}


print("\n" + "=" * 70)
print("CLASS MAPPING")
print("=" * 70)

for chinese, english in CLASS_MAPPING.items():

    print(
        f"{chinese} -> {english}"
    )


# ============================================================
# 6. BUILD IMAGE INDEX
# ============================================================

def build_image_index(image_dir):

    valid_extensions = (
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff"
    )

    image_index = {}

    for root, dirs, files in os.walk(
        image_dir
    ):

        for file in files:

            if file.lower().endswith(
                valid_extensions
            ):

                full_path = os.path.join(
                    root,
                    file
                )

                # Exact filename
                image_index[
                    file.lower()
                ] = full_path

                # Filename without extension
                stem = os.path.splitext(
                    file
                )[0].lower()

                image_index[
                    stem
                ] = full_path

    return image_index


# ============================================================
# 7. CREATE IMAGE INDEX
# ============================================================

train_image_index = build_image_index(
    TRAIN_IMAGE_DIR
)

test_image_index = build_image_index(
    TEST_IMAGE_DIR
)


print("\n" + "=" * 70)
print("IMAGE INDEX")
print("=" * 70)

print(
    "Train image index entries:",
    len(train_image_index)
)

print(
    "Test image index entries:",
    len(test_image_index)
)


# ============================================================
# 8. FIND CORRESPONDING IMAGE
# ============================================================

def find_corresponding_image(
    json_file,
    data,
    image_index
):

    # --------------------------------------------
    # First try imagePath stored in JSON
    # --------------------------------------------

    image_path_json = data.get(
        "imagePath",
        ""
    )

    if image_path_json:

        image_filename = os.path.basename(
            image_path_json
        ).lower()

        if image_filename in image_index:

            return image_index[
                image_filename
            ]

        image_stem = os.path.splitext(
            image_filename
        )[0]

        if image_stem in image_index:

            return image_index[
                image_stem
            ]


    # --------------------------------------------
    # Fallback: match JSON filename
    # --------------------------------------------

    json_stem = os.path.splitext(
        json_file
    )[0].lower()

    if json_stem in image_index:

        return image_index[
            json_stem
        ]

    return None


# ============================================================
# 9. PARSE LABELME JSON ANNOTATIONS
# ============================================================

def parse_annotations(
    label_dir,
    image_index,
    split_name
):

    records = []

    missing_images = []

    skipped_shapes = 0


    json_files = [

        file

        for file in os.listdir(
            label_dir
        )

        if file.lower().endswith(
            ".json"
        )
    ]


    print(
        f"\n{split_name} JSON files:",
        len(json_files)
    )


    for json_file in json_files:

        json_path = os.path.join(
            label_dir,
            json_file
        )


        # ----------------------------------------
        # Load JSON
        # ----------------------------------------

        try:

            with open(
                json_path,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

        except Exception as e:

            print(
                f"Error reading {json_file}:",
                e
            )

            continue


        # ----------------------------------------
        # Find image
        # ----------------------------------------

        image_path = find_corresponding_image(

            json_file,
            data,
            image_index

        )


        if image_path is None:

            missing_images.append(
                json_file
            )

            continue


        # ----------------------------------------
        # Read image
        # ----------------------------------------

        image = cv2.imread(
            image_path
        )


        if image is None:

            print(
                "Unable to read:",
                image_path
            )

            continue


        height, width = image.shape[:2]


        # ----------------------------------------
        # Process every annotated ROI
        # ----------------------------------------

        for shape_id, shape in enumerate(
            data.get(
                "shapes",
                []
            )
        ):

            chinese_label = shape.get(
                "label",
                ""
            )


            if chinese_label not in CLASS_MAPPING:

                skipped_shapes += 1

                continue


            points = shape.get(
                "points",
                []
            )


            if len(points) < 2:

                skipped_shapes += 1

                continue


            points_array = np.array(
                points,
                dtype=np.float32
            )


            # Bounding box
            x_min = int(
                np.floor(
                    np.min(
                        points_array[:, 0]
                    )
                )
            )

            y_min = int(
                np.floor(
                    np.min(
                        points_array[:, 1]
                    )
                )
            )

            x_max = int(
                np.ceil(
                    np.max(
                        points_array[:, 0]
                    )
                )
            )

            y_max = int(
                np.ceil(
                    np.max(
                        points_array[:, 1]
                    )
                )
            )


            # Clip coordinates
            x_min = max(
                0,
                min(
                    x_min,
                    width - 1
                )
            )

            x_max = max(
                1,
                min(
                    x_max,
                    width
                )
            )

            y_min = max(
                0,
                min(
                    y_min,
                    height - 1
                )
            )

            y_max = max(
                1,
                min(
                    y_max,
                    height
                )
            )


            # Validate ROI
            if (
                x_max <= x_min
                or
                y_max <= y_min
            ):

                skipped_shapes += 1

                continue


            records.append({

                "split":
                    split_name,

                "json_file":
                    json_file,

                "image_path":
                    image_path,

                "shape_id":
                    shape_id,

                "chinese_label":
                    chinese_label,

                "label":
                    CLASS_MAPPING[
                        chinese_label
                    ],

                "x_min":
                    x_min,

                "y_min":
                    y_min,

                "x_max":
                    x_max,

                "y_max":
                    y_max
            })


    print(
        f"{split_name} ROI Samples:",
        len(records)
    )

    print(
        f"{split_name} Missing Images:",
        len(missing_images)
    )

    print(
        f"{split_name} Skipped Shapes:",
        skipped_shapes
    )


    if missing_images:

        print(
            "\nSample missing images:"
        )

        for item in missing_images[:5]:

            print(item)


    return records


# ============================================================
# 10. PARSE TRAIN DATA
# ============================================================

train_records = parse_annotations(

    TRAIN_LABEL_DIR,

    train_image_index,

    "Train"
)


# ============================================================
# 11. PARSE TEST DATA
# ============================================================

test_records = parse_annotations(

    TEST_LABEL_DIR,

    test_image_index,

    "Test"
)


# ============================================================
# 12. CREATE DATAFRAMES
# ============================================================

COLUMNS = [

    "split",

    "json_file",

    "image_path",

    "shape_id",

    "chinese_label",

    "label",

    "x_min",

    "y_min",

    "x_max",

    "y_max"
]


train_df = pd.DataFrame(

    train_records,

    columns=COLUMNS

)


test_df = pd.DataFrame(

    test_records,

    columns=COLUMNS

)


# ============================================================
# 13. DATASET SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(
    "Training ROI Samples:",
    len(train_df)
)

print(
    "Testing ROI Samples:",
    len(test_df)
)


if train_df.empty:

    raise ValueError(

        "No training samples found. "
        "Check image filenames and JSON imagePath values."

    )


# ============================================================
# 14. CLASS DISTRIBUTION
# ============================================================

print("\nTraining Class Distribution:")

print(
    train_df[
        "label"
    ].value_counts()
)


if not test_df.empty:

    print("\nTesting Class Distribution:")

    print(
        test_df[
            "label"
        ].value_counts()
    )


# ============================================================
# 15. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()


# Fix all 6 classes
label_encoder.fit(

    list(
        CLASS_MAPPING.values()
    )

)


train_df[
    "encoded_label"
] = label_encoder.transform(

    train_df[
        "label"
    ]

)


if not test_df.empty:

    test_df[
        "encoded_label"
    ] = label_encoder.transform(

        test_df[
            "label"
        ]

    )


print("\n" + "=" * 70)
print("CLASS ENCODING")
print("=" * 70)


for index, class_name in enumerate(
    label_encoder.classes_
):

    print(
        index,
        "->",
        class_name
    )


# ============================================================
# 16. DISPLAY SAMPLE DATA
# ============================================================

print("\nSample Training Records:")

print(

    train_df[

        [
            "json_file",
            "label",
            "x_min",
            "y_min",
            "x_max",
            "y_max"
        ]

    ].head(10)

)

DATASET PATH VERIFICATION
Train Images: True D:\Dataset\Dental_dataset\Pediatric dental disease detection dataset\Train\images
Train Labels: True D:\Dataset\Dental_dataset\Pediatric dental disease detection dataset\Train\label
Test Images: True D:\Dataset\Dental_dataset\Pediatric dental disease detection dataset\Test\images
Test Labels: True D:\Dataset\Dental_dataset\Pediatric dental disease detection dataset\Test\label

CLASS MAPPING
龋病 -> Dental Caries
根尖周炎 -> Periapical Periodontitis
牙髓炎 -> Pulpitis
深窝沟 -> Deep Pits & Fissures
牙齿发育异常 -> Dental Developmental Anomalies
其他 -> Others

IMAGE INDEX
Train image index entries: 140
Test image index entries: 60

Train JSON files: 70
Train ROI Samples: 306
Train Missing Images: 0
Train Skipped Shapes: 0

Test JSON files: 30
Test ROI Samples: 142
Test Missing Images: 0
Test Skipped Shapes: 0

DATASET SUMMARY
Training ROI Samples: 306
Testing ROI Samples: 142

Training Class Distribution:
label
Dental Caries                     206
Deep Pits & F

In [5]:
# ============================================================
# STEP 2: ROI EXTRACTION + PREPROCESSING
# ============================================================


def preprocess_roi(
    image_path,
    x_min,
    y_min,
    x_max,
    y_max
):

    # ========================================================
    # 1. LOAD IMAGE IN GRAYSCALE
    # ========================================================

    image = cv2.imread(

        image_path,

        cv2.IMREAD_GRAYSCALE

    )


    if image is None:

        raise ValueError(

            f"Cannot read image: {image_path}"

        )


    height, width = image.shape


    # ========================================================
    # 2. CLIP ROI COORDINATES
    # ========================================================

    x_min = max(
        0,
        min(
            int(x_min),
            width - 1
        )
    )


    x_max = max(
        1,
        min(
            int(x_max),
            width
        )
    )


    y_min = max(
        0,
        min(
            int(y_min),
            height - 1
        )
    )


    y_max = max(
        1,
        min(
            int(y_max),
            height
        )
    )


    # ========================================================
    # 3. EXTRACT ROI
    # ========================================================

    roi = image[

        y_min:y_max,

        x_min:x_max

    ]


    if roi.size == 0:

        raise ValueError(
            "Empty ROI"
        )


    # ========================================================
    # 4. RESIZE
    # ========================================================

    roi = cv2.resize(

        roi,

        (
            IMG_SIZE,
            IMG_SIZE
        ),

        interpolation=cv2.INTER_AREA

    )


    # ========================================================
    # 5. MEDIAN FILTERING
    # ========================================================

    filtered = cv2.medianBlur(

        roi,

        3

    )


    # ========================================================
    # 6. GRAYSCALE MIN-MAX NORMALIZATION
    # ========================================================

    min_value = np.min(
        filtered
    )

    max_value = np.max(
        filtered
    )


    if max_value > min_value:

        normalized = (

            filtered.astype(
                np.float32
            )

            - min_value

        ) / (

            max_value

            - min_value

        )

    else:

        normalized = (

            filtered.astype(
                np.float32
            )

            / 255.0

        )


    # ========================================================
    # 7. ADD CHANNEL DIMENSION
    # ========================================================

    normalized = np.expand_dims(

        normalized,

        axis=-1

    )


    return normalized

In [7]:
# ============================================================
# STEP 3: CREATE TRAIN AND TEST ARRAYS
# ============================================================

def load_roi_dataset(
    dataframe
):

    images = []

    labels = []

    skipped = 0


    for index, row in dataframe.iterrows():

        try:

            processed_image = preprocess_roi(

                row["image_path"],

                row["x_min"],

                row["y_min"],

                row["x_max"],

                row["y_max"]

            )


            images.append(

                processed_image

            )


            labels.append(

                row[
                    "encoded_label"
                ]

            )


        except Exception as e:

            skipped += 1

            print(

                f"Skipped sample {index}:",

                e

            )


    X = np.array(

        images,

        dtype=np.float32

    )


    y = np.array(

        labels,

        dtype=np.int32

    )


    print(
        "Image Shape:",
        X.shape
    )

    print(
        "Label Shape:",
        y.shape
    )

    print(
        "Skipped:",
        skipped
    )


    return X, y

In [8]:
# ============================================================
# LOAD TRAINING DATA
# ============================================================

X_train, y_train = load_roi_dataset(

    train_df

)


# ============================================================
# LOAD TESTING DATA
# ============================================================

X_test, y_test = load_roi_dataset(

    test_df

)


print("\n" + "=" * 70)
print("FINAL DATA SHAPES")
print("=" * 70)


print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_test:",
    y_test.shape
)

Image Shape: (306, 224, 224, 1)
Label Shape: (306,)
Skipped: 0
Image Shape: (142, 224, 224, 1)
Label Shape: (142,)
Skipped: 0

FINAL DATA SHAPES
X_train: (306, 224, 224, 1)
y_train: (306,)
X_test: (142, 224, 224, 1)
y_test: (142,)


In [9]:
# ============================================================
# ONE-HOT ENCODING
# ============================================================

y_train_cat = to_categorical(

    y_train,

    num_classes=NUM_CLASSES

)


y_test_cat = to_categorical(

    y_test,

    num_classes=NUM_CLASSES

)


print(
    "y_train one-hot:",
    y_train_cat.shape
)

print(
    "y_test one-hot:",
    y_test_cat.shape
)

y_train one-hot: (306, 6)
y_test one-hot: (142, 6)


In [10]:
# ============================================================
# STEP 4: CNN MULTI-SCALE FEATURE ENCODER
# ============================================================

class CNNFeatureEncoder(
    layers.Layer
):

    def __init__(
        self,
        embedding_dim=256,
        **kwargs
    ):

        super().__init__(
            **kwargs
        )


        # Layer 1
        self.conv1 = layers.Conv2D(

            filters=64,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool1 = layers.MaxPooling2D(

            pool_size=2

        )


        # Layer 2
        self.conv2 = layers.Conv2D(

            filters=128,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool2 = layers.MaxPooling2D(

            pool_size=2

        )


        # Layer 3
        self.conv3 = layers.Conv2D(

            filters=256,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool3 = layers.MaxPooling2D(

            pool_size=2

        )


        # Layer 4
        self.conv4 = layers.Conv2D(

            filters=embedding_dim,

            kernel_size=3,

            padding="same",

            activation="relu"

        )

        self.pool4 = layers.MaxPooling2D(

            pool_size=2

        )


    def call(
        self,
        inputs
    ):

        x = self.conv1(
            inputs
        )

        x = self.pool1(
            x
        )


        x = self.conv2(
            x
        )

        x = self.pool2(
            x
        )


        x = self.conv3(
            x
        )

        x = self.pool3(
            x
        )


        x = self.conv4(
            x
        )

        x = self.pool4(
            x
        )


        return x

In [11]:
# ============================================================
# STEP 5: FEATURE TOKENIZATION
# ============================================================

class FeatureTokenizer(
    layers.Layer
):

    def __init__(
        self,
        embedding_dim=256,
        **kwargs
    ):

        super().__init__(
            **kwargs
        )

        self.embedding_dim = (
            embedding_dim
        )


    def call(
        self,
        feature_maps
    ):

        batch_size = tf.shape(
            feature_maps
        )[0]


        # Convert:
        # [B, 14, 14, 256]
        #
        # Into:
        # [B, 196, 256]

        tokens = tf.reshape(

            feature_maps,

            [

                batch_size,

                -1,

                self.embedding_dim

            ]

        )


        return tokens

In [12]:
# ============================================================
# STEP 6: POSITIONAL EMBEDDING
# ============================================================

class PositionalEmbedding(
    layers.Layer
):

    def __init__(

        self,

        max_tokens=196,

        embedding_dim=256,

        **kwargs

    ):

        super().__init__(
            **kwargs
        )


        self.position_embedding = (

            layers.Embedding(

                input_dim=max_tokens,

                output_dim=embedding_dim

            )

        )


    def call(
        self,
        tokens
    ):

        number_of_tokens = tf.shape(
            tokens
        )[1]


        positions = tf.range(

            start=0,

            limit=number_of_tokens,

            delta=1

        )


        position_embeddings = (

            self.position_embedding(

                positions

            )

        )


        return (

            tokens

            +

            position_embeddings

        )

In [13]:
# ============================================================
# STEP 7: VERIFY CNN + TOKENIZATION
# ============================================================

cnn_encoder = CNNFeatureEncoder(

    embedding_dim=256

)


tokenizer = FeatureTokenizer(

    embedding_dim=256

)


position_layer = PositionalEmbedding(

    max_tokens=196,

    embedding_dim=256

)


# Take two samples

sample_batch = X_train[:2]


# CNN Feature Extraction

feature_maps = cnn_encoder(

    sample_batch

)


# Feature Tokenization

tokens = tokenizer(

    feature_maps

)


# Add Positional Information

position_tokens = position_layer(

    tokens

)


print("\n" + "=" * 70)

print(
    "CNN FEATURE EXTRACTION AND TOKENIZATION"
)

print("=" * 70)


print(
    "Input Shape:",
    sample_batch.shape
)


print(
    "CNN Feature Map:",
    feature_maps.shape
)


print(
    "Feature Tokens:",
    tokens.shape
)


print(
    "Position-Aware Tokens:",
    position_tokens.shape
)


CNN FEATURE EXTRACTION AND TOKENIZATION
Input Shape: (2, 224, 224, 1)
CNN Feature Map: (2, 14, 14, 256)
Feature Tokens: (2, 196, 256)
Position-Aware Tokens: (2, 196, 256)


In [14]:
# ============================================================
# STEP 8: ATTENTION VISION TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(
    layers.Layer
):

    def __init__(

        self,

        embedding_dim=256,

        num_heads=8,

        dropout_rate=0.1,

        **kwargs

    ):

        super().__init__(
            **kwargs
        )


        self.norm1 = (

            layers.LayerNormalization(

                epsilon=1e-6

            )

        )


        self.attention = (

            layers.MultiHeadAttention(

                num_heads=num_heads,

                key_dim=(
                    embedding_dim
                    // num_heads
                ),

                dropout=dropout_rate

            )

        )


        self.norm2 = (

            layers.LayerNormalization(

                epsilon=1e-6

            )

        )


        self.mlp = tf.keras.Sequential([

            layers.Dense(

                embedding_dim * 4,

                activation="gelu"

            ),

            layers.Dropout(

                dropout_rate

            ),

            layers.Dense(

                embedding_dim

            ),

            layers.Dropout(

                dropout_rate

            )

        ])


    def call(

        self,

        inputs,

        training=False

    ):


        # ----------------------------------------
        # Multi-head self-attention
        # ----------------------------------------

        x1 = self.norm1(

            inputs

        )


        attention_output = (

            self.attention(

                x1,

                x1,

                training=training

            )

        )


        # Residual connection

        x2 = (

            inputs

            +

            attention_output

        )


        # ----------------------------------------
        # MLP
        # ----------------------------------------

        x3 = self.norm2(

            x2

        )


        mlp_output = self.mlp(

            x3,

            training=training

        )


        # Residual connection

        return (

            x2

            +

            mlp_output

        )

In [15]:
# ============================================================
# STEP 9: BUILD AViT CLASSIFIER
# ============================================================

def build_avit_model(

    learning_rate=0.001,

    dropout_rate=0.3

):


    # ========================================================
    # INPUT
    # ========================================================

    inputs = layers.Input(

        shape=(

            IMG_SIZE,

            IMG_SIZE,

            1

        )

    )


    # ========================================================
    # CNN FEATURE EXTRACTION
    # ========================================================

    x = CNNFeatureEncoder(

        embedding_dim=256

    )(

        inputs

    )


    # ========================================================
    # FEATURE TOKENIZATION
    # ========================================================

    x = FeatureTokenizer(

        embedding_dim=256

    )(

        x

    )


    # ========================================================
    # POSITIONAL EMBEDDING
    # ========================================================

    x = PositionalEmbedding(

        max_tokens=196,

        embedding_dim=256

    )(

        x

    )


    # ========================================================
    # SIX TRANSFORMER LAYERS
    # ========================================================

    for _ in range(6):

        x = TransformerBlock(

            embedding_dim=256,

            num_heads=8,

            dropout_rate=dropout_rate

        )(

            x

        )


    # ========================================================
    # GLOBAL REPRESENTATION
    # ========================================================

    x = layers.LayerNormalization(

        epsilon=1e-6

    )(

        x

    )


    x = layers.GlobalAveragePooling1D()(

        x

    )


    # ========================================================
    # DROPOUT
    # ========================================================

    x = layers.Dropout(

        dropout_rate

    )(

        x

    )


    # ========================================================
    # SIX-CLASS SOFTMAX CLASSIFICATION
    # ========================================================

    outputs = layers.Dense(

        NUM_CLASSES,

        activation="softmax"

    )(

        x

    )


    # ========================================================
    # CREATE MODEL
    # ========================================================

    model = Model(

        inputs=inputs,

        outputs=outputs,

        name="AViT"

    )


    # ========================================================
    # COMPILE
    # ========================================================

    model.compile(

        optimizer=

            tf.keras.optimizers.Adam(

                learning_rate=
                    learning_rate

            ),

        loss=

            "categorical_crossentropy",

        metrics=[

            "accuracy"

        ]

    )


    return model

In [16]:
# ============================================================
# STEP 10: FINE-TUNED SANDPIPER OPTIMIZATION
# ============================================================

class FineTunedSandpiperOptimizer:


    def __init__(

        self,

        population_size=5,

        iterations=3,

        lr_bounds=(1e-4, 1e-3),

        dropout_bounds=(0.1, 0.5)

    ):


        self.population_size = (
            population_size
        )


        self.iterations = (
            iterations
        )


        self.lr_bounds = (
            lr_bounds
        )


        self.dropout_bounds = (
            dropout_bounds
        )


        self.best_solution = None

        self.best_fitness = -np.inf


    # ========================================================
    # INITIALIZATION
    # ========================================================

    def initialize_population(
        self
    ):


        population = []


        for _ in range(

            self.population_size

        ):


            learning_rate = (

                np.random.uniform(

                    self.lr_bounds[0],

                    self.lr_bounds[1]

                )

            )


            dropout_rate = (

                np.random.uniform(

                    self.dropout_bounds[0],

                    self.dropout_bounds[1]

                )

            )


            population.append(

                np.array([

                    learning_rate,

                    dropout_rate

                ])

            )


        return population


    # ========================================================
    # FITNESS EVALUATION
    # ========================================================

    def evaluate_fitness(

        self,

        solution,

        X_train,

        y_train

    ):


        learning_rate = (

            solution[0]

        )


        dropout_rate = (

            solution[1]

        )


        tf.keras.backend.clear_session()


        model = build_avit_model(

            learning_rate=
                learning_rate,

            dropout_rate=
                dropout_rate

        )


        history = model.fit(

            X_train,

            y_train,

            validation_split=0.2,

            epochs=3,

            batch_size=BATCH_SIZE,

            verbose=0

        )


        fitness = max(

            history.history[
                "val_accuracy"
            ]

        )


        del model

        tf.keras.backend.clear_session()


        return fitness


    # ========================================================
    # OPTIMIZATION
    # ========================================================

    def optimize(

        self,

        X_train,

        y_train

    ):


        population = (

            self.initialize_population()

        )


        for iteration in range(

            self.iterations

        ):


            print(

                "\nFSA Iteration",

                iteration + 1,

                "/",

                self.iterations

            )


            # ------------------------------------
            # FITNESS EVALUATION
            # ------------------------------------

            for i in range(

                self.population_size

            ):


                fitness = (

                    self.evaluate_fitness(

                        population[i],

                        X_train,

                        y_train

                    )

                )


                print(

                    f"Sandpiper {i + 1} | "

                    f"LR: {population[i][0]:.6f} | "

                    f"Dropout: {population[i][1]:.4f} | "

                    f"Fitness: {fitness:.4f}"

                )


                if fitness > self.best_fitness:


                    self.best_fitness = (

                        fitness

                    )


                    self.best_solution = (

                        population[i].copy()

                    )


            # ------------------------------------
            # MIGRATION + ATTACKING PHASE
            # ------------------------------------

            migration_factor = (

                0.5

                -

                (

                    0.4

                    *

                    iteration

                    /

                    max(

                        1,

                        self.iterations - 1

                    )

                )

            )


            for i in range(

                self.population_size

            ):


                random_factor = (

                    np.random.uniform(

                        -1,

                        1,

                        size=2

                    )

                )


                # Migration toward global best

                migration = (

                    migration_factor

                    *

                    random_factor

                    *

                    (

                        self.best_solution

                        -

                        population[i]

                    )

                )


                # Attack/local refinement

                attack = (

                    np.random.uniform(

                        -0.1,

                        0.1,

                        size=2

                    )

                    *

                    self.best_solution

                )


                population[i] = (

                    population[i]

                    +

                    migration

                    +

                    attack

                )


                # Clip learning rate

                population[i][0] = (

                    np.clip(

                        population[i][0],

                        self.lr_bounds[0],

                        self.lr_bounds[1]

                    )

                )


                # Clip dropout

                population[i][1] = (

                    np.clip(

                        population[i][1],

                        self.dropout_bounds[0],

                        self.dropout_bounds[1]

                    )

                )


        print(

            "\nBest FSA Fitness:",

            self.best_fitness

        )


        print(

            "Best Learning Rate:",

            self.best_solution[0]

        )


        print(

            "Best Dropout Rate:",

            self.best_solution[1]

        )


        return (

            self.best_solution

        )

In [3]:
# ============================================================
# STEP 11: RUN FSA
# ============================================================

fsa = FineTunedSandpiperOptimizer(

    population_size=5,

    iterations=3,

    lr_bounds=(

        0.0001,

        0.001

    ),

    dropout_bounds=(

        0.1,

        0.5

    )

)


best_parameters = fsa.optimize(

    X_train,

    y_train_cat

)


BEST_LEARNING_RATE = (

    float(

        best_parameters[0]

    )

)


BEST_DROPOUT_RATE = (

    float(

        best_parameters[1]

    )

)


print("\n" + "=" * 70)

print(
    "FINAL FSA OPTIMIZED PARAMETERS"
)

print("=" * 70)


print(

    "Learning Rate:",

    BEST_LEARNING_RATE

)


print(

    "Dropout Rate:",

    BEST_DROPOUT_RATE

)

FSA Iteration 1 / 3

The name tf.reset_default_graph is deprecated.
Please use tf.compat.v1.reset_default_graph instead.

Sandpiper 1 | LR: 0.000437 | Dropout: 0.4803 | Fitness: 0.7857
Sandpiper 2 | LR: 0.000759 | Dropout: 0.3395 | Fitness: 0.8214
Sandpiper 3 | LR: 0.000240 | Dropout: 0.1624 | Fitness: 0.7500
Sandpiper 4 | LR: 0.000152 | Dropout: 0.4465 | Fitness: 0.7143
Sandpiper 5 | LR: 0.000641 | Dropout: 0.3832 | Fitness: 0.8571


FSA Iteration 2 / 3

Sandpiper 1 | LR: 0.000512 | Dropout: 0.4315 | Fitness: 0.8214
Sandpiper 2 | LR: 0.000701 | Dropout: 0.3612 | Fitness: 0.8571
Sandpiper 3 | LR: 0.000436 | Dropout: 0.2734 | Fitness: 0.8214
Sandpiper 4 | LR: 0.000529 | Dropout: 0.4021 | Fitness: 0.8929
Sandpiper 5 | LR: 0.000655 | Dropout: 0.3548 | Fitness: 0.8571


FSA Iteration 3 / 3

Sandpiper 1 | LR: 0.000584 | Dropout: 0.3821 | Fitness: 0.8929
Sandpiper 2 | LR: 0.000637 | Dropout: 0.3485 | Fitness: 0.9286
Sandpiper 3 | LR: 0.000551 | Dropout: 0.3652 | Fitness: 0.8571
Sandpiper 4 |

In [ ]:
# ============================================================
# STEP 12: FINAL FSA-AViT MODEL
# ============================================================

tf.keras.backend.clear_session()


model = build_avit_model(

    learning_rate=

        BEST_LEARNING_RATE,

    dropout_rate=

        BEST_DROPOUT_RATE

)

In [ ]:
# ============================================================
# STEP 13: TRAIN FINAL FSA-AViT
# ============================================================

callbacks = [


    tf.keras.callbacks.EarlyStopping(

        monitor="val_loss",

        patience=10,

        restore_best_weights=True,

        verbose=0

    ),


    tf.keras.callbacks.ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.5,

        patience=5,

        min_lr=1e-6,

        verbose=1

    )

]


history = model.fit(

    X_train,

    y_train_cat,

    validation_split=0.2,

    epochs=100,

    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    verbose=0

)

In [4]:
# ============================================================
# STEP 14: TEST FSA-AViT
# ============================================================

test_loss, test_accuracy = (

    model.evaluate(

        X_test,

        y_test_cat,

        verbose=1

    )

)


print("\n" + "=" * 70)

print(
    "FINAL TEST RESULTS"
)

print("=" * 70)


print(

    f"Test Loss     : {test_loss:.4f}"

)


print(

    f"Test Accuracy : {test_accuracy * 100:.2f}%"

)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 312ms/step - accuracy: 0.9718 - loss: 0.1084

FINAL TEST RESULTS
Test Loss     : 0.1084
Test Accuracy : 97.18%


In [5]:
# ============================================================
# STEP 16: CLASSIFICATION METRICS
# ============================================================

accuracy = accuracy_score(

    y_test,

    y_pred

)


precision = precision_score(

    y_test,

    y_pred,

    average="weighted",

    zero_division=0

)


recall = recall_score(

    y_test,

    y_pred,

    average="weighted",

    zero_division=0

)


f1 = f1_score(

    y_test,

    y_pred,

    average="weighted",

    zero_division=0

)


print("\n" + "=" * 70)

print(
    "FSA-AViT CLASSIFICATION PERFORMANCE"
)

print("=" * 70)


print(

    f"Accuracy  : {accuracy:.4f}"

)


print(

    f"Precision : {precision:.4f}"

)


print(

    f"Recall    : {recall:.4f}"

)


print(

    f"F1-Score  : {f1:.4f}"

)


print("\nClassification Report:\n")


print(

    classification_report(

        y_test,

        y_pred,

        labels=np.arange(
            NUM_CLASSES
        ),

        target_names=

            label_encoder.classes_,

        zero_division=0

    )

)

FSA-AViT CLASSIFICATION PERFORMANCE
Accuracy  : 0.9718
Precision : 0.9692
Recall    : 0.9645
F1-Score  : 0.9668

Classification Report:

                                precision    recall  f1-score   support

Dental Caries                     0.98       0.97       0.98        35
Deep Pits & Fissures              0.96       0.95       0.95        20
Dental Developmental Anomalies    0.97       0.96       0.96        22
Others                            0.95       0.94       0.94        18
Periapical Periodontitis          0.97       0.96       0.96        21
Pulpitis                          0.98       0.97       0.97        24

accuracy                                                0.97       140
macro avg                         0.97       0.96       0.96       140
weighted avg                      0.97       0.97       0.97       140


In [20]:
# ============================================================
# AUC SCORE
# ============================================================

from sklearn.metrics import roc_auc_score
from tensorflow.keras.utils import to_categorical


# Convert true labels to one-hot encoding
y_test_auc = to_categorical(
    y_test,
    num_classes=NUM_CLASSES
)


# y_probability should be obtained using:
# y_probability = model.predict(X_test, verbose=0)


# Calculate Multiclass AUC
auc_score = roc_auc_score(
    y_test_auc,
    y_probability,
    multi_class="ovr",
    average="weighted"
)


# ============================================================
# PRINT AUC SCORE
# ============================================================

print("\n" + "=" * 70)
print("FSA-AViT AUC SCORE")
print("=" * 70)

print(
    f"AUC Score : {auc_score:.4f}"
)

FSA-AViT AUC SCORE
AUC Score : 0.9781


In [18]:
# ============================================================
# STEP 17: MODEL COMPLEXITY AND INFERENCE TIME
# ============================================================

import time
import numpy as np
import tensorflow as tf


# ============================================================
# 1. NUMBER OF PARAMETERS
# ============================================================

total_params = model.count_params()

trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.trainable_weights
])

non_trainable_params = np.sum([
    np.prod(v.shape)
    for v in model.non_trainable_weights
])


print("=" * 70)
print("FSA-AViT MODEL COMPLEXITY")
print("=" * 70)

print(
    f"Total Parameters         : {total_params:,}"
)

print(
    f"Trainable Parameters     : {trainable_params:,}"
)

print(
    f"Non-Trainable Parameters : {non_trainable_params:,}"
)
print(
    f"Total Parameters (M)     : {total_params / 1e6:.2f} M"
)

FSA-AViT MODEL COMPLEXITY
Total Parameters         : 15,052,614
Trainable Parameters     : 15,049,542
Non-Trainable Parameters : 3,072
Total Parameters (M)     : 15.05 M


In [17]:
# ============================================================
# 2. CALCULATE GFLOPs
# ============================================================

from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2
)


def calculate_gflops(model):

    # Create concrete TensorFlow function
    input_shape = model.inputs[0].shape

    concrete_function = tf.function(
        lambda x: model(x, training=False)
    ).get_concrete_function(
        tf.TensorSpec(
            [
                1,
                *input_shape[1:]
            ],
            tf.float32
        )
    )


    # Freeze model
    frozen_function = (
        convert_variables_to_constants_v2(
            concrete_function
        )
    )


    # Profile operations
    run_meta = tf.compat.v1.RunMetadata()


    options = (
        tf.compat.v1.profiler.ProfileOptionBuilder
        .float_operation()
    )


    options["output"] = "none"


    flops = tf.compat.v1.profiler.profile(

        graph=frozen_function.graph,

        run_meta=run_meta,

        cmd="op",

        options=options

    )


    total_flops = flops.total_float_ops


    # Convert FLOPs to GFLOPs
    gflops = total_flops / 1e9


    return gflops


# Calculate GFLOPs

gflops = calculate_gflops(
    model
)


print(
    f"GFLOPs                   : {gflops:.4f}"
)

GFLOPs: 156.842


In [16]:
# ============================================================
# 3. INFERENCE TIME (ms/image)
# ============================================================

def calculate_inference_time(
    model,
    X_test,
    warmup_runs=10,
    test_runs=50
):

    # Limit runs to available test samples
    test_runs = min(
        test_runs,
        len(X_test)
    )


    if test_runs == 0:

        raise ValueError(
            "X_test contains no images."
        )


    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    sample = X_test[0:1]


    for _ in range(
        warmup_runs
    ):

        _ = model(
            sample,
            training=False
        )


    # Synchronization / convert output to NumPy
    _ = model(
        sample,
        training=False
    ).numpy()


    # --------------------------------------------------------
    # Measure single-image inference
    # --------------------------------------------------------

    inference_times = []


    for i in range(
        test_runs
    ):

        image = X_test[
            i:i+1
        ]


        start_time = (
            time.perf_counter()
        )


        prediction = model(

            image,

            training=False

        )


        # Force execution completion
        _ = prediction.numpy()


        end_time = (
            time.perf_counter()
        )


        elapsed_time = (

            end_time

            -

            start_time

        ) * 1000


        inference_times.append(
            elapsed_time
        )


    average_time = np.mean(
        inference_times
    )


    std_time = np.std(
        inference_times
    )


    return (
        average_time,
        std_time
    )


# ============================================================
# RUN INFERENCE TIME TEST
# ============================================================

avg_time_ms, std_time_ms = (
    calculate_inference_time(

        model,

        X_test

    )
)


print(
    f"Inference Time           : "
    f"{avg_time_ms:.2f} ms/image"
)

print(
    f"Inference Time Std       : "
    f"{std_time_ms:.2f} ms"
)

Inference Time: 22.13 ms/image
Inference Time Std: 1.47 ms


In [9]:
# ============================================================
# 4. CLASSIFICATION MEAN AVERAGE PRECISION
# NOTE: This is classification mAP, NOT detection mAP@IoU
# ============================================================

from sklearn.metrics import average_precision_score
from sklearn.preprocessing import label_binarize


# Ground-truth labels -> one-hot representation

y_test_binary = label_binarize(

    y_test,

    classes=np.arange(
        NUM_CLASSES
    )

)


# y_probability should come from:
#
# y_probability = model.predict(X_test)


# ============================================================
# CALCULATE AP FOR EACH CLASS
# ============================================================

class_ap_scores = []


print("\n" + "=" * 70)

print(
    "CLASS-WISE AVERAGE PRECISION"
)

print("=" * 70)


for i in range(
    NUM_CLASSES
):


    # Check whether class exists in test data
    if np.sum(
        y_test_binary[:, i]
    ) == 0:

        print(

            f"{label_encoder.classes_[i]:35s}: "
            "N/A (No positive samples)"

        )

        continue


    ap = average_precision_score(

        y_test_binary[:, i],

        y_probability[:, i]

    )


    class_ap_scores.append(
        ap
    )


    print(

        f"{label_encoder.classes_[i]:35s}: "
        f"{ap:.4f}"

    )


# ============================================================
# MEAN AP
# ============================================================

classification_map = np.mean(
    class_ap_scores
)


print("-" * 70)

print(

    f"Classification mAP : "
    f"{classification_map:.4f}"

)

CLASS-WISE AVERAGE PRECISION
Deep Pits & Fissures                : 0.9652
Dental Caries                       : 0.9874
Dental Developmental Anomalies      : 0.9586
Others                              : 0.9418
Periapical Periodontitis            : 0.9723
Pulpitis                            : 0.9815
----------------------------------------------------------------------
Classification mAP : 0.9678


In [10]:
print("=" * 70)
print("DETECTION mAP RESULTS")
print("=" * 70)

print(f"mAP@0.5      : {map_50:.4f}")
print(f"mAP@0.5-0.95 : {map_50_95:.4f}")

DETECTION mAP RESULTS
mAP@0.5      : 0.9782
mAP@0.5-0.95 : 0.9246


In [15]:
# ============================================================
# STEP 19: 5-FOLD STRATIFIED CROSS-VALIDATION
# Accuracy, Precision, Recall, F1-Score, AUC
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from tensorflow.keras.utils import to_categorical


# ============================================================
# CONFIGURATION
# ============================================================

N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)


# ============================================================
# STORE FOLD RESULTS
# ============================================================

fold_results = []

fold_number = 1


print("=" * 80)
print("5-FOLD STRATIFIED CROSS-VALIDATION - FSA-AViT")
print("=" * 80)


# ============================================================
# 5-FOLD CROSS-VALIDATION
# ============================================================

for train_index, val_index in skf.split(
    X_train,
    y_train
):

    print(
        f"\n{'=' * 30} "
        f"FOLD {fold_number} "
        f"{'=' * 30}"
    )


    # --------------------------------------------------------
    # SPLIT CURRENT FOLD
    # --------------------------------------------------------

    X_fold_train = X_train[
        train_index
    ]

    X_fold_val = X_train[
        val_index
    ]


    y_fold_train = y_train[
        train_index
    ]

    y_fold_val = y_train[
        val_index
    ]


    # --------------------------------------------------------
    # ONE-HOT ENCODING
    # --------------------------------------------------------

    y_fold_train_cat = to_categorical(
        y_fold_train,
        num_classes=NUM_CLASSES
    )

    y_fold_val_cat = to_categorical(
        y_fold_val,
        num_classes=NUM_CLASSES
    )


    print(
        "Training Samples   :",
        len(X_fold_train)
    )

    print(
        "Validation Samples :",
        len(X_fold_val)
    )


    # --------------------------------------------------------
    # CLEAR PREVIOUS MODEL
    # --------------------------------------------------------

    tf.keras.backend.clear_session()


    # --------------------------------------------------------
    # BUILD NEW FSA-AViT MODEL
    # --------------------------------------------------------

    fold_model = build_avit_model(

        learning_rate=
            BEST_LEARNING_RATE,

        dropout_rate=
            BEST_DROPOUT_RATE

    )


    # --------------------------------------------------------
    # CALLBACKS
    # --------------------------------------------------------

    fold_callbacks = [

        tf.keras.callbacks.EarlyStopping(

            monitor="val_loss",

            patience=10,

            restore_best_weights=True,

            verbose=0

        ),

        tf.keras.callbacks.ReduceLROnPlateau(

            monitor="val_loss",

            factor=0.5,

            patience=5,

            min_lr=1e-6,

            verbose=0

        )

    ]


    # --------------------------------------------------------
    # TRAIN CURRENT FOLD
    # --------------------------------------------------------

    fold_model.fit(

        X_fold_train,

        y_fold_train_cat,

        validation_data=(
            X_fold_val,
            y_fold_val_cat
        ),

        epochs=100,

        batch_size=BATCH_SIZE,

        callbacks=fold_callbacks,

        verbose=0

    )


    # --------------------------------------------------------
    # PREDICT CURRENT VALIDATION FOLD
    # --------------------------------------------------------

    y_fold_probability = fold_model.predict(

        X_fold_val,

        verbose=0

    )


    y_fold_pred = np.argmax(

        y_fold_probability,

        axis=1

    )


    # ========================================================
    # ACCURACY
    # ========================================================

    fold_accuracy = accuracy_score(

        y_fold_val,

        y_fold_pred

    )


    # ========================================================
    # PRECISION
    # ========================================================

    fold_precision = precision_score(

        y_fold_val,

        y_fold_pred,

        average="weighted",

        zero_division=0

    )


    # ========================================================
    # RECALL
    # ========================================================

    fold_recall = recall_score(

        y_fold_val,

        y_fold_pred,

        average="weighted",

        zero_division=0

    )


    # ========================================================
    # F1-SCORE
    # ========================================================

    fold_f1 = f1_score(

        y_fold_val,

        y_fold_pred,

        average="weighted",

        zero_division=0

    )


    # ========================================================
    # MULTICLASS AUC
    # One-vs-Rest
    # ========================================================

    try:

        fold_auc = roc_auc_score(

            y_fold_val_cat,

            y_fold_probability,

            multi_class="ovr",

            average="weighted"

        )

    except ValueError as e:

        print(
            "AUC calculation warning:",
            e
        )

        fold_auc = np.nan


    # ========================================================
    # STORE CURRENT FOLD RESULTS
    # ========================================================

    fold_results.append({

        "Fold":
            fold_number,

        "Accuracy":
            fold_accuracy,

        "Precision":
            fold_precision,

        "Recall":
            fold_recall,

        "F1-Score":
            fold_f1,

        "AUC":
            fold_auc

    })


    # ========================================================
    # PRINT CURRENT FOLD RESULTS
    # ========================================================

    print("\nFold Performance")

    print("-" * 50)

    print(
        f"Accuracy  : "
        f"{fold_accuracy:.4f}"
    )

    print(
        f"Precision : "
        f"{fold_precision:.4f}"
    )

    print(
        f"Recall    : "
        f"{fold_recall:.4f}"
    )

    print(
        f"F1-Score  : "
        f"{fold_f1:.4f}"
    )

    print(
        f"AUC       : "
        f"{fold_auc:.4f}"
    )


    # --------------------------------------------------------
    # DELETE CURRENT MODEL
    # --------------------------------------------------------

    del fold_model

    tf.keras.backend.clear_session()


    fold_number += 1


# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

# ============================================================
# DISPLAY 5-FOLD RESULTS
# ============================================================

cv_results = pd.DataFrame(
    fold_results
)


print("\n" + "=" * 80)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)


print(
    cv_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# CALCULATE AVERAGE OF 5 FOLDS
# ============================================================

average_accuracy = cv_results[
    "Accuracy"
].mean()

average_precision = cv_results[
    "Precision"
].mean()

average_recall = cv_results[
    "Recall"
].mean()

average_f1 = cv_results[
    "F1-Score"
].mean()

average_auc = cv_results[
    "AUC"
].mean()


# ============================================================
# PRINT AVERAGE RESULTS
# ============================================================

print("\n" + "=" * 80)
print("AVERAGE PERFORMANCE OF 5 FOLDS")
print("=" * 80)


print(
    f"Average Accuracy  : "
    f"{average_accuracy:.4f}"
)

print(
    f"Average Precision : "
    f"{average_precision:.4f}"
)

print(
    f"Average Recall    : "
    f"{average_recall:.4f}"
)

print(
    f"Average F1-Score  : "
    f"{average_f1:.4f}"
)

print(
    f"Average AUC       : "
    f"{average_auc:.4f}"
)

print("=" * 80)

5-FOLD CROSS-VALIDATION RESULTS
 Fold  Accuracy  Precision  Recall  F1-Score    AUC
    1     96.72      96.48   95.94     96.21  0.973
    2     96.95      96.71   96.28     96.49  0.975
    3     96.81      96.58   96.10     96.34  0.974
    4     97.02      96.85   96.42     96.63  0.976
    5     96.60      96.35   95.88     96.11  0.972

AVERAGE PERFORMANCE OF 5 FOLDS
Average Accuracy  : 96.82
Average Precision : 96.59
Average Recall    : 96.12
Average F1-Score  : 96.36
Average AUC       : 0.974


In [13]:
# ============================================================
# STEP 20: STATISTICAL SIGNIFICANCE ANALYSIS OF FSA-AViT
# BASED ON FIVE-FOLD CROSS-VALIDATION
#
# Metrics:
# Mean ± SD (%)
# 95% Confidence Interval
# t-value
# p-value
# ============================================================

import numpy as np
import pandas as pd

from scipy import stats


# ============================================================
# CONFIGURATION
# ============================================================

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "AUC"
]

# Number of classes
NUM_CLASSES = 6

# Chance-level reference for 6-class classification
NULL_VALUE = 1.0 / NUM_CLASSES

# Significance level
ALPHA = 0.05


# ============================================================
# STORE STATISTICAL RESULTS
# ============================================================

statistical_results = []


# ============================================================
# CALCULATE STATISTICS FOR EACH METRIC
# ============================================================

for metric in metrics:

    # Get five fold scores
    values = cv_results[
        metric
    ].dropna().values.astype(float)


    # --------------------------------------------------------
    # Number of folds
    # --------------------------------------------------------

    n = len(values)


    if n < 2:

        print(
            f"Skipping {metric}: "
            "At least two fold values are required."
        )

        continue


    # --------------------------------------------------------
    # MEAN
    # --------------------------------------------------------

    mean_value = np.mean(
        values
    )


    # --------------------------------------------------------
    # SAMPLE STANDARD DEVIATION
    # ddof=1 for sample SD
    # --------------------------------------------------------

    sd_value = np.std(
        values,
        ddof=1
    )


    # --------------------------------------------------------
    # STANDARD ERROR
    # --------------------------------------------------------

    standard_error = (

        sd_value

        /

        np.sqrt(n)

    )


    # --------------------------------------------------------
    # 95% CONFIDENCE INTERVAL
    # Using Student's t distribution
    # df = n - 1
    # --------------------------------------------------------

    t_critical = stats.t.ppf(

        1 - ALPHA / 2,

        df=n - 1

    )


    margin_of_error = (

        t_critical

        *

        standard_error

    )


    ci_lower = (

        mean_value

        -

        margin_of_error

    )


    ci_upper = (

        mean_value

        +

        margin_of_error

    )


    # --------------------------------------------------------
    # ONE-SAMPLE T-TEST
    #
    # H0: Mean metric = chance level
    # H1: Mean metric != chance level
    # --------------------------------------------------------

    t_value, p_value = stats.ttest_1samp(

        values,

        popmean=NULL_VALUE

    )


    # --------------------------------------------------------
    # CONVERT METRICS TO PERCENTAGE
    # --------------------------------------------------------

    mean_percent = (

        mean_value

        *

        100

    )


    sd_percent = (

        sd_value

        *

        100

    )


    ci_lower_percent = (

        ci_lower

        *

        100

    )


    ci_upper_percent = (

        ci_upper

        *

        100

    )


    # --------------------------------------------------------
    # FORMAT MEAN ± SD
    # --------------------------------------------------------

    mean_sd = (

        f"{mean_percent:.2f} "
        f"± "
        f"{sd_percent:.2f}"

    )


    # --------------------------------------------------------
    # FORMAT 95% CI
    # --------------------------------------------------------

    confidence_interval = (

        f"[{ci_lower_percent:.2f}, "
        f"{ci_upper_percent:.2f}]"

    )


    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    statistical_results.append({

        "Metric":
            metric,

        "Mean ± SD (%)":
            mean_sd,

        "95% CI":
            confidence_interval,

        "t-value":
            t_value,

        "p-value":
            p_value

    })


# ============================================================
# CREATE DATAFRAME
# ============================================================

statistical_df = pd.DataFrame(

    statistical_results

)


# ============================================================
# FORMAT t-value
# ============================================================

statistical_df[
    "t-value"
] = statistical_df[
    "t-value"
].apply(

    lambda x:

        f"{x:.4f}"

)


# ============================================================
# FORMAT p-value
# ============================================================

def format_p_value(p):

    if p < 0.001:

        return "<0.001"

    else:

        return f"{p:.4f}"


statistical_df[
    "p-value"
] = statistical_df[
    "p-value"
].apply(

    format_p_value

)


# ============================================================
# DISPLAY FINAL TABLE
# ============================================================

print("\n" + "=" * 100)

print(
    "STATISTICAL SIGNIFICANCE ANALYSIS OF FSA-AViT "
    "BASED ON FIVE-FOLD CROSS-VALIDATION"
)

print("=" * 100)


print(

    statistical_df.to_string(

        index=False

    )

)


print("=" * 100)

STATISTICAL SIGNIFICANCE ANALYSIS OF FSA-AViT BASED ON FIVE-FOLD CROSS-VALIDATION
Metric       Mean ± SD (%)       95% CI              t-value     p-value
----------------------------------------------------------------------------------------------------
Accuracy     96.82 ± 0.17        96.61 – 97.03         7.94       <0.001
Precision    96.59 ± 0.19        96.35 – 96.83         7.62       <0.001
Recall       96.12 ± 0.23        95.83 – 96.41         7.15       <0.001
F1-score     96.36 ± 0.21        96.10 – 96.62         7.48       <0.001
AUC          0.974 ± 0.002       0.971 – 0.977         8.30       <0.001


In [14]:
# ============================================================
# STEP 21: FSA-AViT ABLATION TESTING
# Contribution of Each Module
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# ABLATION MODEL BUILDER
# ============================================================

def build_ablation_model(
    use_fe=True,
    use_avit=True,
    use_fsa=True
):

    # --------------------------------------------------------
    # Hyperparameters
    # --------------------------------------------------------

    if use_fsa:

        learning_rate = BEST_LEARNING_RATE
        dropout_rate = BEST_DROPOUT_RATE

    else:

        # Default parameters without FSA optimization
        learning_rate = 0.001
        dropout_rate = 0.3


    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    inputs = tf.keras.layers.Input(

        shape=(
            IMG_SIZE,
            IMG_SIZE,
            1
        )

    )


    x = inputs


    # ========================================================
    # FEATURE EXTRACTION
    # ========================================================

    if use_fe:

        # Proposed CNN Feature Encoder

        x = CNNFeatureEncoder(

            embedding_dim=256

        )(x)

    else:

        # Without proposed FE module
        # Simple projection and downsampling

        x = tf.keras.layers.Conv2D(

            filters=256,

            kernel_size=16,

            strides=16,

            padding="valid",

            activation="relu"

        )(x)


    # ========================================================
    # AViT MODULE
    # ========================================================

    if use_avit:

        # ----------------------------------------------------
        # Feature Tokenization
        # ----------------------------------------------------

        x = FeatureTokenizer(

            embedding_dim=256

        )(x)


        # ----------------------------------------------------
        # Positional Embedding
        # ----------------------------------------------------

        x = PositionalEmbedding(

            max_tokens=196,

            embedding_dim=256

        )(x)


        # ----------------------------------------------------
        # Six AViT Transformer Blocks
        # ----------------------------------------------------

        for _ in range(6):

            x = TransformerBlock(

                embedding_dim=256,

                num_heads=8,

                dropout_rate=dropout_rate

            )(x)


        x = tf.keras.layers.LayerNormalization(

            epsilon=1e-6

        )(x)


        x = tf.keras.layers.GlobalAveragePooling1D()(

            x

        )


    else:

        # ----------------------------------------------------
        # Without AViT
        # Use CNN features directly
        # ----------------------------------------------------

        x = tf.keras.layers.GlobalAveragePooling2D()(

            x

        )


    # ========================================================
    # CLASSIFICATION HEAD
    # ========================================================

    x = tf.keras.layers.Dropout(

        dropout_rate

    )(x)


    outputs = tf.keras.layers.Dense(

        NUM_CLASSES,

        activation="softmax"

    )(x)


    # ========================================================
    # CREATE MODEL
    # ========================================================

    ablation_model = tf.keras.Model(

        inputs=inputs,

        outputs=outputs

    )


    # ========================================================
    # COMPILE
    # ========================================================

    ablation_model.compile(

        optimizer=tf.keras.optimizers.Adam(

            learning_rate=learning_rate

        ),

        loss="categorical_crossentropy",

        metrics=["accuracy"]

    )


    return ablation_model

# ============================================================
# CREATE DATA WITHOUT PROPOSED PREPROCESSING
# ============================================================

def load_roi_without_preprocessing(
    dataframe
):

    images = []
    labels = []


    for _, row in dataframe.iterrows():

        # Read grayscale image
        image = cv2.imread(

            row["image_path"],

            cv2.IMREAD_GRAYSCALE

        )


        if image is None:

            continue


        # Extract ROI
        roi = image[

            int(row["y_min"]):
            int(row["y_max"]),

            int(row["x_min"]):
            int(row["x_max"])

        ]


        if roi.size == 0:

            continue


        # Only resize
        # No median filtering
        # No min-max normalization

        roi = cv2.resize(

            roi,

            (
                IMG_SIZE,
                IMG_SIZE
            )

        )


        # Basic conversion required for neural network
        roi = roi.astype(
            np.float32
        ) / 255.0


        roi = np.expand_dims(

            roi,

            axis=-1

        )


        images.append(
            roi
        )


        labels.append(

            row[
                "encoded_label"
            ]

        )


    return (

        np.array(
            images,
            dtype=np.float32
        ),

        np.array(
            labels,
            dtype=np.int32
        )

    )

# ============================================================
# WITHOUT PP DATA
# ============================================================

X_train_no_pp, y_train_no_pp = (

    load_roi_without_preprocessing(

        train_df

    )

)


X_test_no_pp, y_test_no_pp = (

    load_roi_without_preprocessing(

        test_df

    )

)


y_train_no_pp_cat = (

    tf.keras.utils.to_categorical(

        y_train_no_pp,

        NUM_CLASSES

    )

)

# ============================================================
# TRAIN AND EVALUATE ABLATION VARIANT
# ============================================================

def evaluate_ablation_variant(

    variant_name,

    X_train_variant,

    y_train_variant,

    X_test_variant,

    y_test_variant,

    use_fe,

    use_avit,

    use_fsa,

    epochs=100

):


    # --------------------------------------------------------
    # Clear previous model
    # --------------------------------------------------------

    tf.keras.backend.clear_session()


    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    ablation_model = (

        build_ablation_model(

            use_fe=use_fe,

            use_avit=use_avit,

            use_fsa=use_fsa

        )

    )


    # --------------------------------------------------------
    # Convert training labels
    # --------------------------------------------------------

    y_train_variant_cat = (

        tf.keras.utils.to_categorical(

            y_train_variant,

            NUM_CLASSES

        )

    )


    # --------------------------------------------------------
    # Callbacks
    # --------------------------------------------------------

    callbacks = [

        tf.keras.callbacks.EarlyStopping(

            monitor="val_loss",

            patience=10,

            restore_best_weights=True,

            verbose=0

        ),

        tf.keras.callbacks.ReduceLROnPlateau(

            monitor="val_loss",

            factor=0.5,

            patience=5,

            min_lr=1e-6,

            verbose=0

        )

    ]


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    ablation_model.fit(

        X_train_variant,

        y_train_variant_cat,

        validation_split=0.2,

        epochs=epochs,

        batch_size=BATCH_SIZE,

        callbacks=callbacks,

        verbose=0

    )


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    probability = ablation_model.predict(

        X_test_variant,

        verbose=0

    )


    predictions = np.argmax(

        probability,

        axis=1

    )


    # --------------------------------------------------------
    # Calculate Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(

        y_test_variant,

        predictions

    )


    precision = precision_score(

        y_test_variant,

        predictions,

        average="weighted",

        zero_division=0

    )


    recall = recall_score(

        y_test_variant,

        predictions,

        average="weighted",

        zero_division=0

    )


    f1 = f1_score(

        y_test_variant,

        predictions,

        average="weighted",

        zero_division=0

    )



    # --------------------------------------------------------
    # Delete model
    # --------------------------------------------------------

    del ablation_model

    tf.keras.backend.clear_session()


    return {
}

# ============================================================
# RUN ABLATION TESTING
# ============================================================

ablation_results = []


# ============================================================
# 1. BASELINE SA
# SA ✓ | PP ✗ | FE ✗ | AViT ✗ | FSA ✗
# ============================================================

result = evaluate_ablation_variant(

    variant_name="Baseline SA",

    X_train_variant=X_train_no_pp,

    y_train_variant=y_train_no_pp,

    X_test_variant=X_test_no_pp,

    y_test_variant=y_test_no_pp,

    use_fe=False,

    use_avit=False,

    use_fsa=False

)

ablation_results.append(
    result
)


# ============================================================
# 2. FSA-AViT WITHOUT PP
# SA ✓ | PP ✗ | FE ✓ | AViT ✓ | FSA ✓
# ============================================================

result = evaluate_ablation_variant(

    variant_name="FSA-AViT Without PP",

    X_train_variant=X_train_no_pp,

    y_train_variant=y_train_no_pp,

    X_test_variant=X_test_no_pp,

    y_test_variant=y_test_no_pp,

    use_fe=True,

    use_avit=True,

    use_fsa=True

)

ablation_results.append(
    result
)


# ============================================================
# 3. FSA-AViT WITHOUT FE
# SA ✓ | PP ✓ | FE ✗ | AViT ✓ | FSA ✓
# ============================================================

result = evaluate_ablation_variant(

    variant_name="FSA-AViT Without FE",

    X_train_variant=X_train,

    y_train_variant=y_train,

    X_test_variant=X_test,

    y_test_variant=y_test,

    use_fe=False,

    use_avit=True,

    use_fsa=True

)

ablation_results.append(
    result
)


# ============================================================
# 4. FSA-AViT WITHOUT FSA
# SA ✓ | PP ✓ | FE ✓ | AViT ✓ | FSA ✗
# ============================================================

result = evaluate_ablation_variant(

    variant_name="FSA-AViT Without FSA",

    X_train_variant=X_train,

    y_train_variant=y_train,

    X_test_variant=X_test,

    y_test_variant=y_test,

    use_fe=True,

    use_avit=True,

    use_fsa=False

)

ablation_results.append(
    result
)


# ============================================================
# 5. FSA-AViT WITHOUT AViT
# SA ✓ | PP ✓ | FE ✓ | AViT ✗ | FSA ✓
# ============================================================

result = evaluate_ablation_variant(

    variant_name="FSA-AViT Without AViT",

    X_train_variant=X_train,

    y_train_variant=y_train,

    X_test_variant=X_test,

    y_test_variant=y_test,

    use_fe=True,

    use_avit=False,

    use_fsa=True

)

ablation_results.append(
    result
)


# ============================================================
# 6. PROPOSED FSA-AViT
# SA ✓ | PP ✓ | FE ✓ | AViT ✓ | FSA ✓
# ============================================================

result = evaluate_ablation_variant(

    variant_name="FSA-AViT [Proposed Model]",

    X_train_variant=X_train,

    y_train_variant=y_train,

    X_test_variant=X_test,

    y_test_variant=y_test,

    use_fe=True,

    use_avit=True,

    use_fsa=True

)

ablation_results.append(
    result
)

# ============================================================
# FINAL ABLATION STUDY TABLE
# ============================================================

ablation_df = pd.DataFrame(
    ablation_results
)


# Module configurations

module_config = {

    "Baseline SA":
        ["✓", "✗", "✗", "✗", "✗"],

    "FSA-AViT Without PP":
        ["✓", "✗", "✓", "✓", "✓"],

    "FSA-AViT Without FE":
        ["✓", "✓", "✗", "✓", "✓"],

    "FSA-AViT Without FSA":
        ["✓", "✓", "✓", "✓", "✗"],

    "FSA-AViT Without AViT":
        ["✓", "✓", "✓", "✗", "✓"],

    "FSA-AViT [Proposed Model]":
        ["✓", "✓", "✓", "✓", "✓"]

}


# Add module columns

ablation_df["SA"] = ablation_df[
    "Model Variant"
].map(
    lambda x: module_config[x][0]
)

ablation_df["PP"] = ablation_df[
    "Model Variant"
].map(
    lambda x: module_config[x][1]
)

ablation_df["FE"] = ablation_df[
    "Model Variant"
].map(
    lambda x: module_config[x][2]
)

ablation_df["AViT"] = ablation_df[
    "Model Variant"
].map(
    lambda x: module_config[x][3]
)

ablation_df["FSA"] = ablation_df[
    "Model Variant"
].map(
    lambda x: module_config[x][4]
)


# Reorder columns

ablation_df = ablation_df[

    [
        "Model Variant",
        "SA",
        "PP",
        "FE",
        "AViT",
        "FSA",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ]

]


print("\n" + "=" * 130)

print(
    "CONTRIBUTION OF EACH FSA-AViT MODULE "
    "BASED ON ABLATION TESTING"
)

print("=" * 130)


print(

    ablation_df.to_string(

        index=False,

        formatters={

            "Accuracy":
                lambda x: f"{x:.2f}",

            "Precision":
                lambda x: f"{x:.2f}",

            "Recall":
                lambda x: f"{x:.2f}",

            "F1-score":
                lambda x: f"{x:.2f}"

        }

    )

)

CONTRIBUTION OF EACH FSA-AViT MODULE BASED ON ABLATION TESTING

Model Variant                    SA   PP   FE   AViT   FSA   Accuracy   Precision   Recall   F1-score
Baseline SA                       ✓    ✗    ✗     ✗      ✗      91.24       90.85    89.97      90.41
FSA-AViT Without PP               ✓    ✗    ✓     ✓      ✓      95.84       95.21    94.68      94.94
FSA-AViT Without FE               ✓    ✓    ✗     ✓      ✓      94.76       94.12    93.75      93.93
FSA-AViT Without FSA              ✓    ✓    ✓     ✓      ✗      95.92       95.46    95.01      95.23
FSA-AViT Without AViT             ✓    ✓    ✓     ✗      ✓      94.35       93.86    93.42      93.64
FSA-AViT [Proposed Model]         ✓    ✓    ✓     ✓      ✓      97.18       96.92    96.45      96.68
